In [ ]:
from pathlib import Path
import torch
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small, 
    Qwen3Tokenizer,
    Qwen3Model, 
    QWEN_CONFIG_06_B 
)

## 2.4 Preparing input texts for LLMs

In [2]:
download_qwen3_small (kind="base" , tokenizer_only=True , out_dir="qwen3")

In [3]:
tokenizer_path = Path ("qwen3") / "tokenizer-base.json" 
tokenizer = Qwen3Tokenizer (tokenizer_file_path =tokenizer_path)

In [6]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids_list)
print(text)

for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

Explain large language models.
840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [9]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")

        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

device = get_device()
device = torch.device ("cpu")

Using Apple Silicon GPU (MPS)


In [10]:
download_qwen3_small(kind="base", tokenizer_only=False , out_dir="qwen3")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [ ]:
model_path = Path("qwen3") / "qwen3-0.6B-base.pth" 
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

## 2.6 Understanding the sequential LLM text generation process

In [ ]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

input_tensor = torch.tensor(input_token_ids_list)
input_tensor_fmt = input_tensor.unsqueeze(0)
input_tensor_fmt = input_tensor_fmt.to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)
output_tensor_fmt = output_tensor.squeeze(0)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Number of input tokens: 6
Formatted Output tensor shape: torch.Size([6, 151936])


In [14]:
last_token = output_tensor_fmt[-1] 
print(last_token)
print(torch.argmax(last_token, dim=-1, keepdim=True))
print(tokenizer.decode([20286]))

tensor([ 7.3750,  2.0312,  8.0000,  ..., -2.5469, -2.5469, -2.5469],
       dtype=torch.bfloat16)
tensor([20286])
 Large
